In [19]:
import math
import os
import re

import numpy as np
import pandas as pd
import networkx as nx
from sentence_transformers import SentenceTransformer


DEFAULT_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

C:\Users\timofey\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\timofey\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loadi

# Общая реализая метода:

In [20]:
def normalize_rows(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


def make_ngrams(text: str, n: int = 10, step: int = 10) -> list[str]:
    words = text.split()
    if not words:
        return [""]
    if len(words) <= n:
        return [" ".join(words)]
    return [" ".join(words[i:i + n]) for i in range(0, len(words) - n + 1, step)]


def csr(
    description_text: str,
    concept_title: str,
    model: SentenceTransformer = DEFAULT_MODEL,
    n: int = 10,
    step: int = 10,
    concept_emb: np.ndarray | None = None,
    chunk_embs: np.ndarray | None = None,
) -> float:
    chunks = make_ngrams(description_text, n=n, step=step)

    if chunk_embs is None:
        chunk_embs = np.asarray(model.encode(chunks))
    if concept_emb is None:
        concept_emb = np.asarray(model.encode([concept_title]))[0]

    chunk_embs = normalize_rows(np.asarray(chunk_embs))
    concept_emb = concept_emb / max(np.linalg.norm(concept_emb), 1e-12)

    return float((chunk_embs @ concept_emb).sum())


def cer(
    description_text: str,
    concept_title: str,
    n: int = 10,
    step: int = 10,
    case_sensitive: bool = False,
) -> int:
    chunks = make_ngrams(description_text, n=n, step=step)

    if not case_sensitive:
        concept_title = concept_title.lower()
        chunks = [chunk.lower() for chunk in chunks]

    return sum(concept_title in chunk for chunk in chunks)


def transitive_reduce(g: nx.DiGraph) -> nx.DiGraph:
    if not nx.is_directed_acyclic_graph(g):
        return g
    reduced = nx.transitive_reduction(g)
    out = nx.DiGraph()
    out.add_nodes_from(g.nodes())
    out.add_edges_from(reduced.edges())
    return out


def ask_expert(topic_a: str, topic_b: str, score_ab: float, score_ba: float, prs: float, mode: str) -> int:
    print("\n" + "=" * 80)
    print(f"Пара: {topic_a!r}  <->  {topic_b!r}")
    print(f"{mode.upper()}({topic_a} -> {topic_b}) = {score_ab:.6f}")
    print(f"{mode.upper()}({topic_b} -> {topic_a}) = {score_ba:.6f}")
    print(f"PRS = {prs:.6f}")
    print("Введите решение:")
    print(f"  0 -> {topic_a} prerequisite для {topic_b}   ({topic_a} -> {topic_b})")
    print(f"  1 -> {topic_b} prerequisite для {topic_a}   ({topic_b} -> {topic_a})")
    print("  2 -> остановить разметку")

    while True:
        ans = input("Ваш выбор [0/1/2]: ").strip()
        if ans in {"0", "1", "2"}:
            return int(ans)


def prepare_embeddings(
    topic_to_text: dict[str, str],
    model: SentenceTransformer = DEFAULT_MODEL,
    n: int = 10,
    step: int = 10,
):
    topic_embs = {
        topic: np.asarray(model.encode([topic]))[0]
        for topic in topic_to_text
    }

    chunk_embs = {
        topic: np.asarray(model.encode(make_ngrams(text, n=n, step=step)))
        for topic, text in topic_to_text.items()
    }

    return topic_embs, chunk_embs


def build_ranked_pairs(
    topic_to_text: dict[str, str],
    mode: str = "csr",
    model: SentenceTransformer = DEFAULT_MODEL,
    n: int = 10,
    step: int = 10,
):
    topics = list(topic_to_text.keys())
    topic_embs, topic_chunk_embs = prepare_embeddings(topic_to_text, model=model, n=n, step=step)

    ranked = []
    pair_scores = {}

    for i in range(len(topics)):
        for j in range(i + 1, len(topics)):
            a, b = topics[i], topics[j]
            text_a, text_b = topic_to_text[a], topic_to_text[b]

            if mode == "csr":
                score_ab = csr(
                    description_text=text_a,
                    concept_title=b,
                    model=model,
                    n=n,
                    step=step,
                    concept_emb=topic_embs[b],
                    chunk_embs=topic_chunk_embs[a],
                )
                score_ba = csr(
                    description_text=text_b,
                    concept_title=a,
                    model=model,
                    n=n,
                    step=step,
                    concept_emb=topic_embs[a],
                    chunk_embs=topic_chunk_embs[b],
                )
            else:
                score_ab = cer(text_a, b, n=n, step=step)
                score_ba = cer(text_b, a, n=n, step=step)

            prs = max(score_ab, score_ba)
            ranked.append((prs, a, b))
            pair_scores[(a, b)] = (score_ab, score_ba)

    ranked.sort(reverse=True, key=lambda x: x[0])
    return ranked, pair_scores


def run_ace(
    topic_to_text: dict[str, str],
    mode: str = "csr",
    model: SentenceTransformer = DEFAULT_MODEL,
    n: int = 10,
    step: int = 10,
    top_t_percent: float = 100.0,
):
    ranked_pairs, pair_scores = build_ranked_pairs(
        topic_to_text=topic_to_text,
        mode=mode,
        model=model,
        n=n,
        step=step,
    )

    g = nx.DiGraph()
    g.add_nodes_from(topic_to_text.keys())

    limit = math.ceil(len(ranked_pairs) * top_t_percent / 100)

    for prs, a, b in ranked_pairs[:limit]:
        if nx.has_path(g, a, b) or nx.has_path(g, b, a):
            continue

        score_ab, score_ba = pair_scores[(a, b)]
        ans = ask_expert(a, b, score_ab, score_ba, prs, mode)

        if ans == 2:
            break
        elif ans == 0:
            g.add_edge(a, b)
        else:
            g.add_edge(b, a)

        g = transitive_reduce(g)

    return g, ranked_pairs, pair_scores

# Функции для проведения тестирования и сравнения результатов метрики CSR на данных

In [24]:
def norm_name(s: str) -> str:
    return re.sub(r"\s+", "_", s.strip())


def load_example(csv_file: str, descriptions_dir: str):
    df = pd.read_csv(csv_file)
    df.columns = [c.lower() for c in df.columns]

    concepts = sorted(set(df["concept1"].astype(str)) | set(df["concept2"].astype(str)))

    topic_to_text = {
        c: open(os.path.join(descriptions_dir, f"{norm_name(c)}.txt"), encoding="utf-8").read().strip()
        for c in concepts
    }

    return df, topic_to_text


def test_on_csv(csv_file: str, descriptions_dir: str):
    df, topic_to_text = load_example(csv_file, descriptions_dir)

    topic_embs, chunk_embs = prepare_embeddings(
        topic_to_text=topic_to_text,
        model=DEFAULT_MODEL,
        n=10,
        step=10,
    )

    df["calculated_csr_score"] = [
        csr(
            description_text=topic_to_text[str(row["concept2"])],
            concept_title=str(row["concept1"]),
            model=DEFAULT_MODEL,
            n=10,
            step=10,
            concept_emb=topic_embs[str(row["concept1"])],
            chunk_embs=chunk_embs[str(row["concept2"])],
        )
        for _, row in df.iterrows()
    ]

    df["abs_diff"] = (df["calculated_csr_score"] - df["csr_score"]).abs()

    print(f"Количество пар: {len(df)}")
    print(f"Среднее абсолютное отклонение: {df['abs_diff'].mean():.6f}")
    print(f"Максимальное абсолютное отклонение: {df['abs_diff'].max():.6f}")

    return df, topic_to_text

# Результаты сравнения:

In [25]:
csv_file = r".\EKG-Dataset-main\0-100.csv"
descriptions_dir = r".\EKG-Dataset-main\concept_descriptions"

df_test, topic_to_text = test_on_csv(csv_file, descriptions_dir)

df_test[["concept1", "concept2", "csr_score", "calculated_csr_score", "abs_diff"]]

Количество пар: 100
Среднее абсолютное отклонение: 0.746397
Максимальное абсолютное отклонение: 13.171602


,concept1,concept2,csr_score,calculated_csr_score,abs_diff
0,Multiplying Fractions,Dividing Fractions,29.4010,29.080967,0.320033
1,Multiplying Fractions,Converting Mixed Number and Improper Fractions,26.3829,26.173025,0.209875
2,Adding and Subtracting Fractions,Dividing Fractions,25.4758,25.248873,0.226927
3,Dividing Fractions,Converting between Fractions and Percentages,25.2948,24.845112,0.449688
4,Converting between Fractions and Decimals,Converting between Fractions and Percentages,24.6007,24.048004,0.552696
...,...,...,...,...,...
95,Mental Multiplication and Division,Combining Operations,16.6884,17.294109,0.605709
96,Multiplying and Dividing with Decimals,Ordering Fractions,16.6866,16.282660,0.403940
97,Missing Lengths,Length Scale Factors in Similar Shapes,16.5978,16.722698,0.124898
98,"Rounding to the Nearest Whole (10, 100, etc)",Mental Addition and Subtraction,16.5563,14.283169,2.273131


In [26]:
csv_file = r".\EKG-Dataset-main\600-700.csv"
descriptions_dir = r".\EKG-Dataset-main\concept_descriptions"

df_test, topic_to_text = test_on_csv(csv_file, descriptions_dir)

df_test[["concept1", "concept2", "csr_score", "calculated_csr_score", "abs_diff"]]

Количество пар: 100
Среднее абсолютное отклонение: 0.427687
Максимальное абсолютное отклонение: 2.327201


,concept1,concept2,csr_score,calculated_csr_score,abs_diff
0,Adding and Subtracting with Decimals,Laws of Indices,8.8283,9.109797,0.281497
1,Writing Expressions,Ordering Negative Numbers,8.8276,9.484349,0.656749
2,Direct Proportion,Two Way Tables,8.8200,8.722605,0.097395
3,Mental Addition and Subtraction,Averages and Range from Frequency Table,8.8187,9.409244,0.590544
4,Place Value,Mixed operation Decimals,8.7780,9.077299,0.299299
...,...,...,...,...,...
95,Area of Simple Shapes,Naming Co-ordinates in 2D,8.2114,8.750909,0.539509
96,Range and Interquartile Range from a List of Data,Pie Chart,8.1975,8.131556,0.065944
97,Writing Ratios,Range and Interquartile Range from a List of Data,8.1972,8.778930,0.581730
98,Converting between Fractions and Decimals,Ordering Negative Numbers,8.1836,8.493641,0.310041


In [27]:
csv_file = r".\EKG-Dataset-main\200-300.csv"
descriptions_dir = r".\EKG-Dataset-main\concept_descriptions"

df_test, topic_to_text = test_on_csv(csv_file, descriptions_dir)

df_test[["concept1", "concept2", "csr_score", "calculated_csr_score", "abs_diff"]]

Количество пар: 100
Среднее абсолютное отклонение: 0.491056
Максимальное абсолютное отклонение: 2.737585


,concept1,concept2,csr_score,calculated_csr_score,abs_diff
0,Mental Addition and Subtraction,Written Division,13.8300,13.863512,0.033512
1,Writing Expressions,BIDMAS,13.7963,14.232243,0.435943
2,Percentages of an Amount,Simplifying and Equivalent Ratios,13.7544,13.409493,0.344907
3,Multiplying and Dividing Negative Numbers,Expanding Single Brackets,13.7357,15.267788,1.532088
4,Written Multiplication,Converting between Fractions and Percentages,13.7322,13.734874,0.002674
...,...,...,...,...,...
95,Multiplying Fractions,Adding and Subtracting Negative Numbers,12.1751,12.532859,0.357759
96,Multiplying and Dividing with Decimals,"Squares, Cubes, etc",12.1402,12.567971,0.427771
97,Written Subtraction,Rounding to Decimal Places,12.1037,12.438309,0.334609
98,Multiplying and Dividing with Decimals,Laws of Indices,12.0981,12.838152,0.740052
